# THIS IS STRICTLY FOR SERVING ON KAGGLE DUAL/SINGLE T4 (PREFERRED DUAL) (FOR TESTING)

In [1]:
# CELL 1: ENVIRONMENT SETUP & DEPENDENCIES
import os
import subprocess
import sys

print("[1/5] Installing dependencies and sanitizing container environment...")

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf<6.0.0,>=5.26.1",
    "pyngrok>=7.0.0",
    "fastapi",
    "uvicorn",
    "httpx",
    "chromadb",
    "sentence-transformers",
    "huggingface_hub",
    "nvidia-ml-py",
    "rapidfuzz",
    "vllm",
    "psutil",
]
subprocess.run(cmd, check=True)

# Remove torchcodec to prevent vLLM C-extension dynamic linking errors
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchcodec"],
    stderr=subprocess.DEVNULL,
)

print("✓ Core dependencies installed and C-extensions sanitized.")

[1/5] Installing dependencies and sanitizing container environment...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.85.1 requires protobuf<8.0.0,>=7.35.1, but you have protobuf 5.29.6 which is incompatible.
kaggle-environments 1.33.3 requires protobuf>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
onnx 1.23.2 requires protobuf>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-texttospeech 2.38.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-vision 3.16.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 wh

Found existing installation: torchcodec 0.17.0
Uninstalling torchcodec-0.17.0:
  Successfully uninstalled torchcodec-0.17.0
✓ Core dependencies installed and C-extensions sanitized.


In [2]:
# CELL 2: GIT SYNC (feat/v2-vela3-awq-rag) & CLEAN PRE-FLIGHT ASSET CACHE
import json
import os
import shutil
from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import CrossEncoder, SentenceTransformer

REPO_URL = "https://github.com/AhmedH32/ai-gym-coach.git"
BRANCH_NAME = "feat/v2-vela3-awq-rag"
PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()

print(f"[2/5] Syncing {REPO_URL} (branch: {BRANCH_NAME})...")
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH_NAME} {REPO_URL} /kaggle/working/ai-gym-coach
else:
    %cd /kaggle/working/ai-gym-coach
    !git fetch --all
    !git checkout {BRANCH_NAME}
    !git reset --hard origin/{BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

%cd /kaggle/working/ai-gym-coach

# 1. Clear any corrupted snapshot cache of vela3-fast to guarantee a fresh copy
hf_cache_dir = Path("/root/.cache/huggingface/hub")
for d in hf_cache_dir.glob("models--vllm-sr--vela3-fast*"):
    shutil.rmtree(d, ignore_errors=True)
print("✓ Cleared stale vela3-fast hub cache.")

# 2. Pre-cache Bi-Encoder, Cross-Encoder, and Vela 3.0 Fast DecisionEncoder
print("Pre-caching models and applying clean FP16 Tensor Core patch...")
_ = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cpu")
_ = CrossEncoder("BAAI/bge-reranker-base", device="cpu")
vela_dir = Path(
    snapshot_download(
        repo_id="vllm-sr/vela3-fast",
        allow_patterns=["*.py", "*.json", "*.txt", "model.safetensors", "tokenizer*"],
    )
)

# 3. Deterministic line-by-line patch for FP16 Tensor Core execution on sm_75
vela_enc_file = vela_dir / "vela3_encoder.py"
if vela_enc_file.exists():
    raw_lines = vela_enc_file.read_text(encoding="utf-8").splitlines()
    clean_lines = []
    for line in raw_lines:
        if "amp = torch.autocast(" in line:
            clean_lines.append('        amp = torch.autocast("cuda", dtype=torch.float16, enabled=str(self.device).startswith("cuda"))')
        else:
            clean_lines.append(line)
    vela_enc_file.write_text("\n".join(clean_lines), encoding="utf-8")
    print("✓ Successfully applied clean FP16 Tensor Core patch to vela3_encoder.py.")

print("✓ All models cached and pre-flight assertions passed.")

[2/5] Syncing https://github.com/AhmedH32/ai-gym-coach.git (branch: feat/v2-vela3-awq-rag)...
Cloning into '/kaggle/working/ai-gym-coach'...
remote: Enumerating objects: 5627, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (259/259), done.
remote: Total 5627 (delta 138), reused 262 (delta 83), pack-reused 5270 (from 2)
Receiving objects: 100% (5627/5627), 115.91 MiB | 35.30 MiB/s, done.
Resolving deltas: 100% (309/309), done.
/kaggle/working/ai-gym-coach
✓ Cleared stale vela3-fast hub cache.
Pre-caching models and applying clean FP16 Tensor Core patch...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (17 outputs left out)

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✓ Successfully applied clean FP16 Tensor Core patch to vela3_encoder.py.
✓ All models cached and pre-flight assertions passed.


In [3]:
import importlib
import backend.rag_engine.vector_store
importlib.reload(backend.rag_engine.vector_store)
from backend.rag_engine.vector_store import RAGVectorStore

print("Re-indexing ChromaDB with high-density colloquial search surfaces...")
store = RAGVectorStore(device="cuda:1")
stats = store.ingest_all_corpora(
    injuries_dir="/kaggle/working/ai-gym-coach/rag_corpus/injuries",
    periodization_dir="/kaggle/working/ai-gym-coach/rag_corpus/periodization"
)
print(f"✓ Re-index stats: {stats}")

Re-indexing ChromaDB with high-density colloquial search surfaces...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Re-index stats: {'upserted': 40, 'deleted': 0, 'unchanged': 0, 'total_active': 40}


In [4]:
# CELL 3: LAUNCH OPTIMIZED AWQ vLLM SERVER ON GPU 0 (PORT 8001)
import os
import subprocess
import sys
import time
from pathlib import Path
import psutil
import requests

for proc in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmd_str = " ".join(proc.info["cmdline"] or [])
        if "vllm" in cmd_str or "8001" in cmd_str:
            proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
time.sleep(2)

PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()
vllm_log_path = Path("/kaggle/working/vllm_awq.log")
with open(vllm_log_path, "w") as f:
    f.write("--- vLLM AWQ Startup Log ---\n")

# Strictly isolate vLLM to GPU 0 so GPU 1 remains 100% free for the System-1 & RAG Co-Processor
vllm_env = os.environ.copy()
vllm_env["CUDA_VISIBLE_DEVICES"] = "0"

vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--dtype", "half",
    "--gpu-memory-utilization", "0.78",
    "--max-model-len", "8192",
    "--enable-prefix-caching",
]

print("[3/5] Launching Qwen/Qwen2.5-7B-Instruct-AWQ (CUDA Graphs Enabled, 8k Context) on GPU 0...")
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=open(vllm_log_path, "a"),
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
    env=vllm_env,
)

vllm_ready = False
log_pos = 0

for _ in range(120):  # Up to 10 minutes for initial AWQ weight download + CUDA graph capture
    if vllm_log_path.exists():
        with open(vllm_log_path, "r") as f:
            f.seek(log_pos)
            new_lines = f.read()
            if new_lines:
                for line in new_lines.strip().split("\n"):
                    if any(k in line for k in [
                        "Loading", "quantization", "KV cache", "Graph capturing",
                        "Capturing CUDA", "Application startup complete", "ERROR"
                    ]):
                        print(f"  [vLLM-AWQ] {line}")
                log_pos = f.tell()

    try:
        r = requests.get("http://127.0.0.1:8001/v1/models", timeout=2)
        if r.status_code == 200:
            vllm_ready = True
            models = [m["id"] for m in r.json().get("data", [])]
            print(f"\n✓ [vLLM-AWQ LIVE ON GPU 0] Models loaded: {models}")
            break
    except Exception:
        pass

    if vllm_proc.poll() is not None:
        print("\n❌ FATAL: vLLM-AWQ exited unexpectedly! Full log:")
        print(vllm_log_path.read_text())
        raise RuntimeError("vLLM-AWQ failed to start.")

    time.sleep(5)

if not vllm_ready:
    print(vllm_log_path.read_text())
    raise TimeoutError("vLLM-AWQ failed to report ready within 10 minutes.")

[3/5] Launching Qwen/Qwen2.5-7B-Instruct-AWQ (CUDA Graphs Enabled, 8k Context) on GPU 0...
  [vLLM-AWQ] (EngineCore pid=354) INFO 11-08 04:52:19 [core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_awq, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, rea

In [5]:
# CELL 4: SELF-CONTAINED BENCHMARK SUITE (STEP 3 SINGLE-T4 vs STEP 4 DUAL-T4) & FASTAPI LAUNCH
import asyncio
import gc
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path
from typing import Any, Dict, List

import httpx
import numpy as np
import psutil
import requests
import torch

PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.agent_core.orchestrator import CONTRASTIVE_FEW_SHOTS, GymCoachOrchestrator
from backend.agent_core.vela_router import VelaSystemOneRouter
from backend.analytics_engine.recovery import AnalyticsEngine
from backend.rag_engine.tools import ClinicalRAGTools

VLLM_URL = "http://127.0.0.1:8001/v1"
FASTAPI_URL = "http://127.0.0.1:8000"
AWQ_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
OUTPUT_JSON_PATH = Path("/kaggle/working/complete_before_after_telemetry.json")

# ==============================================================================
# 1. EMBEDDED STEP-1 LEGACY BASELINE TELEMETRY (Captured on BnB NF4 + Eager + LoRA)
# ==============================================================================
LEGACY_STEP1_BASELINE = {
    "architecture": "LEGACY_BNB_NF4_EAGER_LORA_32K",
    "vram_telemetry": {
        "gpu_0": {"static_baseline_mb": 14237.19, "peak_used_mb": 14397.19, "vram_utilization_pct": 93.73, "peak_compute_util_pct": 100.0},
        "gpu_1": {"static_baseline_mb": 451.19, "peak_used_mb": 451.19, "vram_utilization_pct": 2.94, "peak_compute_util_pct": 0.0},
    },
    "rag_legacy_telemetry": {
        "recall_at_1": 0.5417,
        "recall_at_3": 0.5833,
        "mrr": 0.5625,
        "false_rejection_rate_at_tau_038": 0.3333,
        "ood_rejection_rate_orchestrator_layer": 0.0,
        "orchestrator_card_drop_rate": 1.0,
        "latency_p50_ms": 85.06,
        "latency_p95_ms": 115.24,
    },
    "pass1_lora_router_telemetry": {
        "p50_latency_ms": 15779.5,
        "p95_latency_ms": 31360.15,
        "mean_latency_ms": 19038.0,
    },
    "pass2_streaming_probes": {
        "cold_prefix_ttft_ms": 27108.51,
        "warm_prefix_ttft_ms": 646.13,
        "mean_itl_ms": 124.75,
        "mean_decode_tps": 8.28,
    },
    "e2e_pipeline_telemetry": {
        "p50_latency_s": 105.391,
        "p95_latency_s": 115.101,
        "max_latency_s": 116.192,
        "mean_latency_s": 84.961,
    },
}


# ==============================================================================
# 2. DUAL-GPU NVML VRAM PROFILER & STREAMING SSE PROBE HELPERS
# ==============================================================================
class DualGPUVRAMProfiler:
    def __init__(self, interval_sec: float = 0.05):
        self.interval = interval_sec
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None
        self.samples: List[Dict[str, Any]] = []
        self.use_nvml = False
        try:
            import pynvml
            pynvml.nvmlInit()
            self.pynvml = pynvml
            self.device_count = pynvml.nvmlDeviceGetCount()
            self.handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(self.device_count)]
            self.use_nvml = True
        except Exception:
            self.device_count = torch.cuda.device_count()

    def _read_gpu_memory_mb(self) -> List[Dict[str, float]]:
        stats = []
        if self.use_nvml:
            for idx, h in enumerate(self.handles):
                mem = self.pynvml.nvmlDeviceGetMemoryInfo(h)
                util = self.pynvml.nvmlDeviceGetUtilizationRates(h)
                stats.append({
                    "gpu_id": idx,
                    "used_mb": round(mem.used / (1024 ** 2), 2),
                    "free_mb": round(mem.free / (1024 ** 2), 2),
                    "total_mb": round(mem.total / (1024 ** 2), 2),
                    "gpu_util_pct": float(util.gpu),
                })
        else:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.used,memory.free,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
                text=True,
            )
            for line in out.strip().splitlines():
                parts = [float(x.strip()) for x in line.split(",")]
                stats.append({
                    "gpu_id": int(parts[0]),
                    "used_mb": parts[1],
                    "free_mb": parts[2],
                    "total_mb": parts[3],
                    "gpu_util_pct": parts[4],
                })
        return stats

    def _loop(self) -> None:
        while not self._stop_event.is_set():
            try:
                self.samples.append({"ts": time.perf_counter(), "gpus": self._read_gpu_memory_mb()})
            except Exception:
                pass
            time.sleep(self.interval)

    def start(self) -> List[Dict[str, float]]:
        self.samples.clear()
        self._stop_event.clear()
        baseline = self._read_gpu_memory_mb()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return baseline

    def stop(self) -> Dict[str, Any]:
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=2.0)
        if not self.samples:
            return {}
        summary = {}
        num_gpus = len(self.samples[0]["gpus"])
        for g_idx in range(num_gpus):
            used_series = [s["gpus"][g_idx]["used_mb"] for s in self.samples]
            util_series = [s["gpus"][g_idx]["gpu_util_pct"] for s in self.samples]
            total_mb = self.samples[0]["gpus"][g_idx]["total_mb"]
            summary[f"gpu_{g_idx}"] = {
                "total_vram_mb": total_mb,
                "static_baseline_mb": used_series[0],
                "peak_used_mb": round(float(np.max(used_series)), 2),
                "mean_used_mb": round(float(np.mean(used_series)), 2),
                "dynamic_spike_delta_mb": round(float(np.max(used_series) - used_series[0]), 2),
                "vram_utilization_pct": round((float(np.max(used_series)) / total_mb) * 100.0, 2),
                "peak_compute_util_pct": round(float(np.max(util_series)), 1),
                "mean_compute_util_pct": round(float(np.mean(util_series)), 1),
            }
        return summary


def scrape_vllm_prometheus_metrics() -> Dict[str, float]:
    metrics_out: Dict[str, float] = {}
    try:
        resp = requests.get("http://127.0.0.1:8001/metrics", timeout=3.0)
        if resp.status_code == 200:
            for line in resp.text.splitlines():
                if line.startswith("#") or not line.strip():
                    continue
                for key in [
                    "vllm:gpu_cache_usage_perc",
                    "vllm:prefix_cache_queries_total",
                    "vllm:prefix_cache_hits_total",
                    "vllm:num_preemptions_total",
                ]:
                    if line.startswith(key):
                        metrics_out[key] = float(line.split()[-1])
    except Exception:
        pass
    return metrics_out


async def stream_vllm_completion(
    client: httpx.AsyncClient,
    model: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 128,
    temperature: float = 0.0,
) -> Dict[str, Any]:
    url = f"{VLLM_URL}/chat/completions"
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    t_start = time.perf_counter()
    t_first_token: float | None = None
    token_timestamps: List[float] = []
    chunks_text: List[str] = []
    usage_data: Dict[str, int] = {}

    async with client.stream("POST", url, json=payload, timeout=180.0) as response:
        async for line in response.aiter_lines():
            if not line or not line.startswith("data: "):
                continue
            data_str = line[6:].strip()
            if data_str == "[DONE]":
                break
            try:
                chunk = json.loads(data_str)
                if chunk.get("usage"):
                    usage_data = chunk["usage"]
                choices = chunk.get("choices", [])
                if choices:
                    delta_content = choices[0].get("delta", {}).get("content")
                    if delta_content:
                        now = time.perf_counter()
                        if t_first_token is None:
                            t_first_token = now
                        token_timestamps.append(now)
                        chunks_text.append(delta_content)
            except Exception:
                continue

    t_end = time.perf_counter()
    total_latency_s = t_end - t_start
    ttft_ms = ((t_first_token - t_start) * 1000.0) if t_first_token else (total_latency_s * 1000.0)
    completion_tokens = usage_data.get("completion_tokens", len(token_timestamps))
    prompt_tokens = usage_data.get("prompt_tokens", 0)
    decode_time_s = (t_end - t_first_token) if (t_first_token and completion_tokens > 1) else 0.001

    if len(token_timestamps) > 1:
        itl_diffs_ms = np.diff(token_timestamps) * 1000.0
        mean_itl_ms = float(np.mean(itl_diffs_ms))
        p95_itl_ms = float(np.percentile(itl_diffs_ms, 95))
    else:
        mean_itl_ms = (decode_time_s * 1000.0) / max(1, completion_tokens - 1)
        p95_itl_ms = mean_itl_ms

    tok_per_sec = completion_tokens / max(0.001, decode_time_s) if completion_tokens > 1 else 0.0
    return {
        "total_latency_s": round(total_latency_s, 3),
        "ttft_ms": round(ttft_ms, 2),
        "mean_itl_ms": round(mean_itl_ms, 2),
        "p95_itl_ms": round(p95_itl_ms, 2),
        "decode_tps": round(tok_per_sec, 2),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }


# ==============================================================================
# 3. EVALUATION DATASETS (RAG N=30, ROUTER N=12, E2E N=5)
# ==============================================================================
COLLOQUIAL_RAG_TEST_SET = [
    {"query": "Front of my shoulder pinches sharp whenever I bench press or do upright rows", "expected_id": "COND_SUBACROMIAL_IMPINGEMENT"},
    {"query": "Tendon right below my kneecap is barking when I squat deep out of the hole", "expected_id": "COND_PATELLAR_TENDINOPATHY"},
    {"query": "Outside of my elbow burns when I grip heavy deadlifts or do reverse curls", "expected_id": "COND_LATERAL_EPICONDYLITIS"},
    {"query": "Inside of my elbow aches every time I do heavy underhand pulldowns or wrist curls", "expected_id": "COND_MEDIAL_EPICONDYLITIS"},
    {"query": "Sharp pain shooting down the back of my buttock when sitting hard or rotating my hip", "expected_id": "COND_PIRIFORMIS_SYNDROME"},
    {"query": "Bottom of my heel stabs with pain on my first steps out of bed and during calf raises", "expected_id": "COND_PLANTAR_FASCIITIS"},
    {"query": "Front of my hip pinches deep in the socket when I squat ass to grass", "expected_id": "COND_FEMOROACETABULAR_IMPINGEMENT"},
    {"query": "Lower back hurts when I round forward on heavy deadlifts or sit-ups", "expected_id": "COND_LUMBAR_DISC_HERNIATION"},
    {"query": "Low back aches whenever I arch hard during standing overhead press or back extensions", "expected_id": "COND_LUMBAR_FACET_SYNDROME"},
    {"query": "Back of my heel cord is stiff and sore after box jumps and jump rope", "expected_id": "COND_ACHILLES_TENDINOPATHY"},
    {"query": "Outer side of my knee rubs and hurts when running downhill or doing narrow leg press", "expected_id": "COND_ILIOTIBIAL_BAND_SYNDROME"},
    {"query": "Inner side of my shins ache badly after high volume box jumps and running", "expected_id": "COND_MEDIAL_TIBIAL_STRESS_SYNDROME"},
    {"query": "Deep ache under my kneecap going down stairs or doing heavy leg extensions", "expected_id": "COND_PATELLOFEMORAL_PAIN_SYNDROME"},
    {"query": "Top bump of my shoulder hurts when doing pec deck flyes or deep dips across my chest", "expected_id": "COND_AC_JOINT_SPRAIN"},
    {"query": "Shoulder feels like it wants to slip out the front when I do heavy dumbbell flyes or behind neck press", "expected_id": "COND_ANTERIOR_SHOULDER_INSTABILITY"},
    {"query": "Shoulder is completely stiff and frozen, cannot raise my arm overhead or rotate it out", "expected_id": "COND_ADHESIVE_CAPSULITIS"},
    {"query": "Pulled my inner thigh groin doing wide sumo deadlifts and side lunges", "expected_id": "COND_ADDUCTOR_STRAIN"},
    {"query": "Front of my upper arm grooves hurts during incline dumbbell curls and preacher curls", "expected_id": "COND_BICEPS_TENDINOPATHY"},
    {"query": "Lower bicep crease at the elbow hurts when locking out heavy curls or mixed grip deadlifts", "expected_id": "COND_DISTAL_BICEPS_TENDINOPATHY"},
    {"query": "Pinky and ring finger tingle when my elbow stays bent on heavy skullcrushers or preacher pads", "expected_id": "COND_CUBITAL_TUNNEL_SYNDROME"},
    {"query": "Pointy bone on the back tip of my elbow is swollen and hurts when doing forearm planks", "expected_id": "COND_OLECRANON_BURSITIS"},
    {"query": "Back of my elbow tendon hurts locking out skull crushers and heavy dips", "expected_id": "COND_TRICEPS_TENDINOPATHY"},
    {"query": "Outer bone of my hip hurts when lying on my side or letting my hip drop on single leg squats", "expected_id": "COND_GREATER_TROCHANTERIC_PAIN_SYNDROME"},
    {"query": "Deep click and pain inside my shoulder when doing kipping pull-ups or heavy overhead snatches", "expected_id": "COND_SLAP_LESION"},
    # 6 Out-Of-Domain / Acute Medical Emergency Queries
    {"query": "Crushing tightness in my left chest radiating to my jaw and shortness of breath", "expected_id": "OUT_OF_DOMAIN"},
    {"query": "Heard a loud audible pop in my pec on bench press and my chest muscle balled up with purple bruising", "expected_id": "OUT_OF_DOMAIN"},
    {"query": "Sudden loss of bladder control and complete saddle numbness in both legs after deadlifting", "expected_id": "OUT_OF_DOMAIN"},
    {"query": "Dizzy, blurred vision, and spinning room vertigo whenever I stand up from the bench", "expected_id": "OUT_OF_DOMAIN"},
    {"query": "Sharp lower right abdominal pain with fever and nausea since this morning", "expected_id": "OUT_OF_DOMAIN"},
    {"query": "How do I configure Docker compose volumes for a PostgreSQL database?", "expected_id": "OUT_OF_DOMAIN"},
]

ROUTER_TEST_SUITE = [
    {"query": "Crushing tightness in my left chest radiating to my jaw and shortness of breath", "expected_intent": "CLASS_0_RED_FLAG", "expected_workout": False},
    {"query": "Heard a loud audible pop in my pec on bench press and my chest muscle balled up with purple bruising", "expected_intent": "CLASS_0_RED_FLAG", "expected_workout": False},
    {"query": "Sudden loss of bladder control and complete saddle numbness in both legs after deadlifting", "expected_intent": "CLASS_0_RED_FLAG", "expected_workout": False},
    {"query": "Tendon below my kneecap is barking on deep squats, but I have dumbbells. Can I still get a leg workout in today?", "expected_intent": "CLASS_1_CLINICAL_RAG", "expected_workout": True},
    {"query": "Felt a sharp pinching pain in the front of my shoulder during flat bench press yesterday. What happened?", "expected_intent": "CLASS_1_CLINICAL_RAG", "expected_workout": False},
    {"query": "Outside of my elbow burns when I grip heavy deadlifts, give me an arm session I can do today", "expected_intent": "CLASS_1_CLINICAL_RAG", "expected_workout": True},
    {"query": "Build me a hypertrophy back and biceps workout using only dumbbells and cables, no barbell.", "expected_intent": "CLASS_2_CATALOG_WORKOUT", "expected_workout": True},
    {"query": "Need a chest and tris pump at home with just my rusty 20s dumbbells, no machines.", "expected_intent": "CLASS_2_CATALOG_WORKOUT", "expected_workout": True},
    {"query": "Give me a quad and glute leg day using machines and cables, don't bend me over with lower back hinges.", "expected_intent": "CLASS_2_CATALOG_WORKOUT", "expected_workout": True},
    {"query": "My barbell bench press 1RM has stalled for 3 weeks straight at 100kg and my joints feel beat up. How should I structure a deload?", "expected_intent": "CLASS_3_DIRECT_COACHING", "expected_workout": False},
    {"query": "How many weekly hard sets do I need to stay in optimal MAV for hypertrophy without exceeding MRV?", "expected_intent": "CLASS_3_DIRECT_COACHING", "expected_workout": False},
    {"query": "Should I tuck my elbows at 45 degrees or flare them out during the eccentric phase of a barbell bench press?", "expected_intent": "CLASS_3_DIRECT_COACHING", "expected_workout": False},
]

E2E_BENCHMARK_QUERIES = [
    {"tag": "CLASS_1_INJURY_WORKOUT", "query": "Tendon below my kneecap is barking on deep squats, but I have dumbbells. Can I still get a leg workout in today?"},
    {"tag": "CLASS_1_INJURY_DIAGNOSTIC", "query": "Felt a sharp pinching pain in the front of my shoulder during flat bench press yesterday. What happened?"},
    {"tag": "CLASS_2_CATALOG_WORKOUT", "query": "Build me a hypertrophy back and biceps workout using only dumbbells and cables, no barbell."},
    {"tag": "CLASS_3_DIRECT_COACHING", "query": "My barbell bench press 1RM has stalled for 3 weeks straight at 100kg and my joints feel beat up. How should I structure a deload?"},
    {"tag": "CLASS_0_RED_FLAG_EMERGENCY", "query": "Heard a loud audible pop in my pec on bench press and my chest muscle balled up with purple bruising"},
]


# ==============================================================================
# 4. CO-PROCESSOR PLACEMENT BENCHMARK (CUDA:0 CO-LOCATED vs CUDA:1 DISAGGREGATED)
# ==============================================================================
async def benchmark_coprocessor_placement(placement_label: str, device_str: str) -> Dict[str, Any]:
    print("\n" + "-" * 85)
    print(f"⚡ BENCHMARKING PLACEMENT: {placement_label} (device='{device_str}')")
    print("-" * 85)

    profiler = DualGPUVRAMProfiler(interval_sec=0.05)
    profiler.start()

    t_init0 = time.perf_counter()
    rag_tools = ClinicalRAGTools(device=device_str)
    router = VelaSystemOneRouter(device=device_str)
    init_s = round(time.perf_counter() - t_init0, 2)

    router_latencies_ms: List[float] = []
    intent_correct = 0
    workout_gate_correct = 0
    router_decisions_log = []

    for pass_idx in range(2):
        for case in ROUTER_TEST_SUITE:
            dec = router.route(case["query"])
            router_latencies_ms.append(dec.latency_ms)
            if pass_idx == 0:
                intent_correct += int(dec.intent == case["expected_intent"])
                workout_gate_correct += int(dec.wants_workout_program == case["expected_workout"])
                router_decisions_log.append({
                    "query": case["query"][:60] + "...",
                    "predicted_intent": dec.intent,
                    "expected_intent": case["expected_intent"],
                    "confidence": dec.intent_confidence,
                    "wants_workout": dec.wants_workout_program,
                    "muscles": dec.target_muscles,
                    "equipment": dec.available_equipment,
                    "excluded": dec.excluded_constraints,
                    "latency_ms": dec.latency_ms,
                })

    in_domain_cases = [c for c in COLLOQUIAL_RAG_TEST_SET if c["expected_id"] != "OUT_OF_DOMAIN"]
    ood_cases = [c for c in COLLOQUIAL_RAG_TEST_SET if c["expected_id"] == "OUT_OF_DOMAIN"]

    # Ablation 1: Dense-Only with Lenient Threshold (tau=0.58, no Cross-Encoder)
    dense_hits_1, dense_hits_3, dense_rr = 0, 0, []
    for case in in_domain_cases:
        d_res = rag_tools.search_medical_db(case["query"], n_results=3, distance_threshold=0.58, use_reranker=False)
        m_ids = d_res.matched_ids
        if m_ids:
            if m_ids[0] == case["expected_id"]:
                dense_hits_1 += 1
            if case["expected_id"] in m_ids[:3]:
                dense_hits_3 += 1
                dense_rr.append(1.0 / (m_ids.index(case["expected_id"]) + 1))
            else:
                dense_rr.append(0.0)
        else:
            dense_rr.append(0.0)

    # Ablation 2: Full Two-Stage (Dense K=10, tau=0.58 + Cross-Encoder Reranker)
    rerank_hits_1, rerank_hits_3, rerank_rr = 0, 0, []
    false_rejections = 0
    total_rag_ms, stage1_ms_list, stage2_ms_list = [], [], []

    for case in in_domain_cases:
        r_res = rag_tools.search_medical_db(
            case["query"], n_results=3, candidate_pool_size=10,
            distance_threshold=0.58, rerank_prob_threshold=0.12, use_reranker=True,
        )
        total_rag_ms.append(r_res.latency_ms)
        stage1_ms_list.append(r_res.stage1_ms)
        stage2_ms_list.append(r_res.rerank_ms)

        if r_res.triage_required or not r_res.matched_ids:
            false_rejections += 1
            rerank_rr.append(0.0)
        else:
            if r_res.matched_ids[0] == case["expected_id"]:
                rerank_hits_1 += 1
            if case["expected_id"] in r_res.matched_ids[:3]:
                rerank_hits_3 += 1
                rerank_rr.append(1.0 / (r_res.matched_ids.index(case["expected_id"]) + 1))
            else:
                rerank_rr.append(0.0)

    ood_rejected_count = 0
    for case in ood_cases:
        r_res = rag_tools.search_medical_db(
            case["query"], n_results=3, candidate_pool_size=10,
            distance_threshold=0.58, rerank_prob_threshold=0.12, use_reranker=True,
        )
        total_rag_ms.append(r_res.latency_ms)
        if r_res.triage_required and r_res.status == "OUT_OF_DOMAIN":
            ood_rejected_count += 1

    vram_stats = profiler.stop()

    summary = {
        "placement_label": placement_label,
        "device": device_str,
        "init_time_s": init_s,
        "vram_telemetry": vram_stats,
        "vela3_router_telemetry": {
            "p50_latency_ms": round(float(np.percentile(router_latencies_ms, 50)), 2),
            "p95_latency_ms": round(float(np.percentile(router_latencies_ms, 95)), 2),
            "mean_latency_ms": round(float(np.mean(router_latencies_ms)), 2),
            "intent_accuracy": round(intent_correct / len(ROUTER_TEST_SUITE), 4),
            "workout_gate_accuracy": round(workout_gate_correct / len(ROUTER_TEST_SUITE), 4),
            "sample_decisions": router_decisions_log[:6],
        },
        "rag_ablation_telemetry": {
            "dense_only_lenient_tau058": {
                "recall_at_1": round(dense_hits_1 / len(in_domain_cases), 4),
                "recall_at_3": round(dense_hits_3 / len(in_domain_cases), 4),
                "mrr": round(float(np.mean(dense_rr)), 4),
            },
            "two_stage_dense_plus_cross_encoder": {
                "recall_at_1": round(rerank_hits_1 / len(in_domain_cases), 4),
                "recall_at_3": round(rerank_hits_3 / len(in_domain_cases), 4),
                "mrr": round(float(np.mean(rerank_rr)), 4),
                "false_rejection_rate": round(false_rejections / len(in_domain_cases), 4),
                "ood_rejection_rate": round(ood_rejected_count / len(ood_cases), 4),
                "orchestrator_card_drop_rate": 0.0,
                "total_latency_p50_ms": round(float(np.percentile(total_rag_ms, 50)), 2),
                "total_latency_p95_ms": round(float(np.percentile(total_rag_ms, 95)), 2),
                "stage1_biencoder_p50_ms": round(float(np.percentile(stage1_ms_list, 50)), 2),
                "stage2_crossencoder_p50_ms": round(float(np.percentile(stage2_ms_list, 50)), 2),
            },
        },
    }

    print(
        f"  • [Vela 3.0 Fast Router on {device_str}] "
        f"p50={summary['vela3_router_telemetry']['p50_latency_ms']}ms | "
        f"p95={summary['vela3_router_telemetry']['p95_latency_ms']}ms | "
        f"Intent Acc={summary['vela3_router_telemetry']['intent_accuracy']*100:.1f}% | "
        f"Workout Gate Acc={summary['vela3_router_telemetry']['workout_gate_accuracy']*100:.1f}%"
    )
    ts_rag = summary["rag_ablation_telemetry"]["two_stage_dense_plus_cross_encoder"]
    print(
        f"  • [RAG Two-Stage + Cross-Encoder on {device_str}] "
        f"Recall@1={ts_rag['recall_at_1']*100:.1f}% | "
        f"Recall@3={ts_rag['recall_at_3']*100:.1f}% | "
        f"MRR={ts_rag['mrr']:.3f} | "
        f"OOD Rejection={ts_rag['ood_rejection_rate']*100:.1f}% | "
        f"p50={ts_rag['total_latency_p50_ms']}ms (Stage1={ts_rag['stage1_biencoder_p50_ms']}ms, Rerank={ts_rag['stage2_crossencoder_p50_ms']}ms)"
    )

    del rag_tools
    del router
    gc.collect()
    torch.cuda.empty_cache()
    return summary


# ==============================================================================
# 5. MAIN EXECUTION: STEP 3, STEP 4, STREAMING PROBES & E2E GATEWAY
# ==============================================================================
async def run_complete_v2_benchmark():
    print("=" * 85)
    print("🚀 STARTING STEP 3 & STEP 4: V2 ARCHITECTURE & DUAL-T4 BENCHMARK SUITE")
    print("=" * 85)

    # 1. Step 3 (Co-Located on GPU 0) vs Step 4 (Disaggregated on GPU 1)
    step3_colocated = await benchmark_coprocessor_placement(
        placement_label="STEP_3_SINGLE_T4_COLOCATED_CUDA0",
        device_str="cuda:0",
    )

    dual_gpu_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
    step4_disaggregated = await benchmark_coprocessor_placement(
        placement_label="STEP_4_DUAL_T4_DISAGGREGATED_CUDA1",
        device_str=dual_gpu_device,
    )

    # 2. Benchmark AWQ + CUDA Graphs Streaming TTFT (3-Tier Radix Cache) & Decode ITL
    print("\n" + "-" * 85)
    print("⚡ BENCHMARKING AWQ + CUDA GRAPHS STREAMING TTFT (3-TIER RADIX CACHE) & ITL")
    print("-" * 85)

    orchestrator = GymCoachOrchestrator(
        vllm_base_url=VLLM_URL,
        base_model_name=AWQ_MODEL,
        project_root=PROJECT_ROOT,
        coprocessor_device=dual_gpu_device,
    )

    session_profile_a = AnalyticsEngine.build_system_context(
        muscle_states=[
            AnalyticsEngine.evaluate_muscle_recovery("Quadriceps", hours_elapsed=48.0, weekly_sets=12),
            AnalyticsEngine.evaluate_muscle_recovery("Pectorals", hours_elapsed=72.0, weekly_sets=14),
        ],
        stalled_exercises=["Barbell_Bench_Press"],
    )

    async with httpx.AsyncClient() as client:
        cold_salt = f"<!-- COLD_SALT_{time.time_ns()} -->\n"
        cold_msgs = [
            {"role": "system", "content": cold_salt + orchestrator.synthesizer_system_prompt},
            *CONTRASTIVE_FEW_SHOTS,
            {"role": "user", "content": f"{session_profile_a}\n\n{E2E_BENCHMARK_QUERIES[0]['query']}"},
        ]
        cold_probe = await stream_vllm_completion(client, model=AWQ_MODEL, messages=cold_msgs, max_tokens=128)

        # Prime Tier-1 + Tier-2 prefix in vLLM Radix tree
        prime_msgs = orchestrator.build_tiered_messages(
            dynamic_turn_prompt="Warmup priming query.",
            session_user_profile=session_profile_a,
        )
        _ = await stream_vllm_completion(client, model=AWQ_MODEL, messages=prime_msgs, max_tokens=8)

        # Warm Tier-1 Prefix Hit (different user profile, shared static system + few-shots)
        session_profile_b = AnalyticsEngine.build_system_context(
            muscle_states=[AnalyticsEngine.evaluate_muscle_recovery("Lats", hours_elapsed=24.0, weekly_sets=16)],
            stalled_exercises=[],
        )
        tier1_warm_msgs = orchestrator.build_tiered_messages(
            dynamic_turn_prompt=E2E_BENCHMARK_QUERIES[1]["query"],
            session_user_profile=session_profile_b,
        )
        tier1_warm_probe = await stream_vllm_completion(client, model=AWQ_MODEL, messages=tier1_warm_msgs, max_tokens=128)

        # Warm Tier-1 + Tier-2 Session Prefix Hit (same user session_profile_a, new turn query)
        tier2_warm_msgs = orchestrator.build_tiered_messages(
            dynamic_turn_prompt=E2E_BENCHMARK_QUERIES[2]["query"],
            session_user_profile=session_profile_a,
        )
        tier2_warm_probe = await stream_vllm_completion(client, model=AWQ_MODEL, messages=tier2_warm_msgs, max_tokens=128)

    mean_awq_itl = round(
        float(np.mean([cold_probe["mean_itl_ms"], tier1_warm_probe["mean_itl_ms"], tier2_warm_probe["mean_itl_ms"]])), 2
    )
    mean_awq_tps = round(
        float(np.mean([cold_probe["decode_tps"], tier1_warm_probe["decode_tps"], tier2_warm_probe["decode_tps"]])), 2
    )
    print(f"  • Cold Prefix TTFT (0% Cache Hit):           {cold_probe['ttft_ms']}ms ({cold_probe['prompt_tokens']} prompt tokens)")
    print(f"  • Warm Tier-1 Prefix TTFT (Static Hit):      {tier1_warm_probe['ttft_ms']}ms | Speedup: {cold_probe['ttft_ms']/max(1.0, tier1_warm_probe['ttft_ms']):.2f}x")
    print(f"  • Warm Tier-2 Session TTFT (User Hit):       {tier2_warm_probe['ttft_ms']}ms | Speedup: {cold_probe['ttft_ms']/max(1.0, tier2_warm_probe['ttft_ms']):.2f}x")
    print(f"  • AWQ + CUDA Graphs Decode Throughput:       {mean_awq_tps} tok/s (ITL: {mean_awq_itl} ms/tok)")

    # 3. Launch Production FastAPI Gateway on Dual-T4 Topology & Run E2E Benchmark
    print("\n" + "-" * 85)
    print("⚡ LAUNCHING FASTAPI v2 GATEWAY (PORT 8000) & RUNNING E2E PIPELINE BENCHMARK")
    print("-" * 85)

    del orchestrator
    gc.collect()
    torch.cuda.empty_cache()

    for proc in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            cmd_str = " ".join(proc.info["cmdline"] or [])
            if "backend.server" in cmd_str or "8000" in cmd_str:
                proc.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    time.sleep(2)

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT)
    env["VLLM_BASE_URL"] = VLLM_URL
    env["BASE_MODEL_NAME"] = AWQ_MODEL
    env["COPROCESSOR_DEVICE"] = dual_gpu_device
    env["ANONYMIZED_TELEMETRY"] = "False"
    env["PYTHONUNBUFFERED"] = "1"

    fastapi_log_path = Path("/kaggle/working/fastapi_v2.log")
    with open(fastapi_log_path, "w") as f:
        f.write("--- FastAPI v2 Startup Log ---\n")

    fastapi_proc = subprocess.Popen(
        [sys.executable, "-m", "backend.server"],
        stdout=open(fastapi_log_path, "a"),
        stderr=subprocess.STDOUT,
        cwd=str(PROJECT_ROOT),
        env=env,
    )

    fastapi_ready = False
    for _ in range(45):
        try:
            r = requests.get(f"{FASTAPI_URL}/health", timeout=2)
            if r.status_code == 200:
                fastapi_ready = True
                print(f"✓ [FastAPI v2 LIVE] Health: {r.json()}")
                break
        except Exception:
            pass
        time.sleep(2)

    if not fastapi_ready:
        print(fastapi_log_path.read_text())
        raise RuntimeError("FastAPI v2 failed to start.")

    e2e_profiler = DualGPUVRAMProfiler(interval_sec=0.1)
    e2e_profiler.start()

    e2e_results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        for item in E2E_BENCHMARK_QUERIES:
            t0 = time.perf_counter()
            resp = await client.post(
                f"{FASTAPI_URL}/api/v1/chat",
                json={"query": item["query"], "session_user_profile": session_profile_a},
            )
            elapsed_s = round(time.perf_counter() - t0, 3)
            body = resp.json() if resp.status_code == 200 else {"error": resp.text}
            tel = body.get("telemetry", {})
            workout_items_cnt = (
                len(body["workout_proposal"]["items"])
                if body.get("workout_proposal") and body["workout_proposal"].get("items")
                else 0
            )
            e2e_results.append({
                "tag": item["tag"],
                "status_code": resp.status_code,
                "e2e_latency_s": elapsed_s,
                "routing_class": body.get("routing_class"),
                "has_workout": body.get("has_workout"),
                "workout_items_count": workout_items_cnt,
                "chat_text_chars": len(body.get("chat_text", "")),
                "telemetry": tel,
            })
            print(
                f"  • [E2E v2 | {item['tag']}] {elapsed_s}s | "
                f"Route: {body.get('routing_class')} | has_workout={body.get('has_workout')} "
                f"(items={workout_items_cnt}) | Router={tel.get('router_latency_ms')}ms | "
                f"RAG={tel.get('rag_latency_ms')}ms | Tokens={tel.get('completion_tokens')}"
            )

    e2e_vram = e2e_profiler.stop()
    prom_metrics = scrape_vllm_prometheus_metrics()

    # Core 4 queries match Step 1's exact 4 E2E queries for direct comparison
    core_4_latencies = [r["e2e_latency_s"] for r in e2e_results[:4]]

    final_report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "legacy_step1_baseline": LEGACY_STEP1_BASELINE,
        "step3_single_t4_colocated": step3_colocated,
        "step4_dual_t4_disaggregated": step4_disaggregated,
        "awq_streaming_probes": {
            "cold_prefix_ttft_ms": cold_probe["ttft_ms"],
            "warm_tier1_prefix_ttft_ms": tier1_warm_probe["ttft_ms"],
            "warm_tier2_session_ttft_ms": tier2_warm_probe["ttft_ms"],
            "mean_itl_ms": mean_awq_itl,
            "mean_decode_tps": mean_awq_tps,
        },
        "e2e_v2_pipeline_telemetry": {
            "core4_p50_latency_s": round(float(np.percentile(core_4_latencies, 50)), 3),
            "core4_p95_latency_s": round(float(np.percentile(core_4_latencies, 95)), 3),
            "core4_max_latency_s": round(float(np.max(core_4_latencies)), 3),
            "core4_mean_latency_s": round(float(np.mean(core_4_latencies)), 3),
            "e2e_vram_telemetry": e2e_vram,
            "vllm_prometheus": prom_metrics,
            "runs": e2e_results,
        },
    }

    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2)

    print("\n" + "=" * 85)
    print(f"✅ BENCHMARK COMPLETE — SAVED TO: {OUTPUT_JSON_PATH}")
    print("=" * 85)
    print(json.dumps(final_report, indent=2))


await run_complete_v2_benchmark()

🚀 STARTING STEP 3 & STEP 4: V2 ARCHITECTURE & DUAL-T4 BENCHMARK SUITE

-------------------------------------------------------------------------------------
⚡ BENCHMARKING PLACEMENT: STEP_3_SINGLE_T4_COLOCATED_CUDA0 (device='cuda:0')
-------------------------------------------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  • [Vela 3.0 Fast Router on cuda:0] p50=125.8ms | p95=129.25ms | Intent Acc=91.7% | Workout Gate Acc=83.3%
  • [RAG Two-Stage + Cross-Encoder on cuda:0] Recall@1=87.5% | Recall@3=100.0% | MRR=0.931 | OOD Rejection=100.0% | p50=61.88ms (Stage1=15.59ms, Rerank=46.2ms)

-------------------------------------------------------------------------------------
⚡ BENCHMARKING PLACEMENT: STEP_4_DUAL_T4_DISAGGREGATED_CUDA1 (device='cuda:1')
-------------------------------------------------------------------------------------


... (6 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✓ Orchestrator indexed 876 catalog exercises.
  • Cold Prefix TTFT (0% Cache Hit):           8040.76ms (3773 prompt tokens)
  • Warm Tier-1 Prefix TTFT (Static Hit):      207.18ms | Speedup: 38.81x
  • Warm Tier-2 Session TTFT (User Hit):       182.6ms | Speedup: 44.03x
  • AWQ + CUDA Graphs Decode Throughput:       35.83 tok/s (ITL: 28.13 ms/tok)

-------------------------------------------------------------------------------------
⚡ LAUNCHING FASTAPI v2 GATEWAY (PORT 8000) & RUNNING E2E PIPELINE BENCHMARK
-------------------------------------------------------------------------------------
✓ [FastAPI v2 LIVE] Health: {'status': 'healthy', 'catalog_size': 876, 'router_model': 'vllm-sr/vela3-fast', 'coprocessor_device': 'cuda:1', 'rag_device': 'cuda:1', 'base_model': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'adapter_model': 'unused_in_system1'}
  • [E2E v2 | CLASS_1_INJURY_WORKOUT] 8.025s | Route: CLASS_A_INJURY | has_workout=False (items=0) | Router=176.75ms | RAG=197.33ms | Tokens=193
  • [E2

In [6]:
# CELL 5 (OPTIONAL): NGROK PUBLIC TUNNEL & LIVE LOG STREAMER
import time
from pathlib import Path
import requests
from pyngrok import ngrok

STATIC_NGROK_DOMAIN = "preshow-spiritual-rink.ngrok-free.dev"

try:
    from kaggle_secrets import UserSecretsClient
    NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_AUTH_TOKEN_HERE"

if not NGROK_AUTH_TOKEN or "PASTE_YOUR" in NGROK_AUTH_TOKEN:
    raise ValueError("Missing NGROK_AUTH_TOKEN! Set it in Kaggle Secrets or Cell 5.")

print(f"[5/5] Binding ngrok static domain: https://{STATIC_NGROK_DOMAIN}...")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
tunnel = ngrok.connect(8000, domain=STATIC_NGROK_DOMAIN)
time.sleep(2)

r = requests.get(f"{tunnel.public_url}/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
print(f"✓ Public gateway health [HTTP {r.status_code}]: {r.json()}")
print("=" * 80)
print(f"🚀 AI GYM COACH v2 PUBLIC API LIVE: POST {tunnel.public_url}/api/v1/chat")
print("=" * 80)

ValueError: Missing NGROK_AUTH_TOKEN! Set it in Kaggle Secrets or Cell 5.